# 04 — Evaluation, Domain Shift Audit, and Advanced Prompt Engineering

**Purpose:** Full post-training evaluation pipeline plus scaffolding for two advanced inference strategies.

**Sections:**
1. Environment setup + generate test-set reports for all variants
2. Full metric stack (BERTScore, CheXbert, BLEU-4, ROUGE-L)
3. Summary table and figures
4. Domain Shift Audit — acquisition shift (image perturbation)
5. Domain Shift Audit — prevalence shift (importance sampling)
6. [SCAFFOLD] RAG retrieval — inject similar training reports into the prompt
7. [SCAFFOLD] Association rules conditioner — soft diagnostic prior via prompt

**Checkpoints compared:**
- Zero-shot MedGemma 4B-it (no fine-tuning)
- `qlora_uniform_v3` — fine-tuned, uniform sampling, BERTScore=0.7113
- `qlora_weighted_v4` — fine-tuned, active weighted sampler (ESS-based p_target)

**STEPs 6–7 can be run immediately** (CPU-only, no checkpoint needed).

## STEP 1 — Environment setup

In [ ]:
import os, sys, json, logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
import yaml

matplotlib.rcParams['figure.dpi'] = 120
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

REPO_ROOT = Path(os.getcwd())
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

PROCESSED_DIR  = REPO_ROOT / 'data' / 'processed'
CHECKPOINT_DIR = REPO_ROOT / 'checkpoints'
FIGURES_DIR    = REPO_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

with open(REPO_ROOT / 'params.yaml') as f:
    params = yaml.safe_load(f)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU    : {props.name}  {props.total_memory/1e9:.1f} GB')

# Load test set
test_df = pd.read_parquet(PROCESSED_DIR / 'test.parquet')
test_df = test_df[test_df['findings'].notna() & (test_df['findings'].str.strip() != '')].reset_index(drop=True)
references = test_df['findings'].str.strip().tolist()
print(f'Test set: {len(test_df)} studies')

# Secrets
env_file = REPO_ROOT / '.env'
if env_file.exists():
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())
    import huggingface_hub
    hf_token = os.environ.get('HF_TOKEN')
    if hf_token:
        huggingface_hub.login(token=hf_token, add_to_git_credential=False)
        print('HF: logged in')

## STEP 2 — Generate test-set reports for all variants

Generates `max_new_tokens=256`, `temperature=0`, greedy decoding for reproducibility.  
Results are cached to `reports/eval_hypotheses_{variant}.json` — re-running the cell skips generation.  
**Expected time per variant: ~25–40 min on Ada RTX 4000 (600 studies).**

In [ ]:
from PIL import Image
from tqdm import tqdm
from src.training.model import load_model_and_processor

IMAGES_DIR = REPO_ROOT / params['data']['images_dir'] / 'images_normalized'
_BLANK = Image.new('RGB', (224, 224), color=(128, 128, 128))

SYSTEM_PROMPT = (
    "You are an expert radiologist. "
    "Write only the Findings section of a radiology report for the chest X-ray shown. "
    "Be concise and clinical. Do not include an Impression section."
)

VARIANTS = {
    'zero_shot':   None,                              # no adapter
    'uniform_v3':  'qlora_uniform_v3/best_model',
    'weighted_v4': 'qlora_weighted_v4/best_model',
}

def _load_image(row):
    frontal = row.get('frontal', [])
    if hasattr(frontal, '__len__') and len(frontal) > 0:
        p = IMAGES_DIR / list(frontal)[0]
        if p.exists():
            try: return Image.open(p).convert('RGB')
            except Exception: pass
    return _BLANK

def _generate_all(model, processor, df, cache_path):
    if cache_path.exists():
        hyps = json.loads(cache_path.read_text())
        print(f'  Loaded {len(hyps)} cached hypotheses from {cache_path.name}')
        return hyps
    hyps = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        img = _load_image(row)
        indication = str(row.get('indication', '') or '').strip()
        if indication.lower() in {'nan', 'none', ''}: indication = ''
        user_text = f'Indication: {indication}\n{SYSTEM_PROMPT}' if indication else SYSTEM_PROMPT
        content = [{'type': 'image'}, {'type': 'text', 'text': user_text}]
        prompt = processor.apply_chat_template(
            [{'role': 'user', 'content': content}],
            add_generation_prompt=True, tokenize=False,
        )
        inputs = processor(text=prompt, images=[img], return_tensors='pt', padding=True)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        with torch.inference_mode():
            out = model.generate(
                **inputs,
                max_new_tokens=params['model']['max_new_tokens'],
                do_sample=False,
                pad_token_id=processor.tokenizer.eos_token_id,
            )
        hyp = processor.tokenizer.decode(
            out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
        ).strip()
        hyps.append(hyp)
    cache_path.write_text(json.dumps(hyps, ensure_ascii=False, indent=2))
    print(f'  Saved {len(hyps)} hypotheses to {cache_path.name}')
    return hyps

hypotheses = {}
for variant, adapter_rel in VARIANTS.items():
    cache = REPO_ROOT / 'reports' / f'eval_hypotheses_{variant}.json'
    print(f'\n[{variant}]')
    # Load base model
    model, processor = load_model_and_processor(
        model_id=params['model']['base_model_id'],
        quantization=params['model']['quantization'],
    )
    # Attach adapter if needed
    if adapter_rel is not None:
        adapter_path = CHECKPOINT_DIR / adapter_rel
        if not adapter_path.exists():
            print(f'  SKIP — checkpoint not found: {adapter_path}')
            del model, processor
            torch.cuda.empty_cache()
            continue
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, str(adapter_path))
    model.eval()
    hypotheses[variant] = _generate_all(model, processor, test_df, cache)
    del model, processor
    torch.cuda.empty_cache()
    print(f'  Done. VRAM freed.')

print(f'\nVariants with hypotheses: {list(hypotheses.keys())}')

## STEP 3 — Full metric stack

Computes for each variant:
- **BERTScore-F1** (primary) — `microsoft/deberta-xlarge-mnli`
- **CheXbert micro/macro F1** (diagnostic)
- **BLEU-4** and **ROUGE-L** (NLG comparability)

Results are cached to `reports/eval_metrics_{variant}.json`.

In [ ]:
import bert_score.utils as _bsu
from bert_score import score as _bert_score
from src.data.labels import CHEXBERT_LABELS, run_chexbert
from sklearn.metrics import f1_score
import evaluate as hf_evaluate

# Monkey-patch for Rust tokenizer overflow (see diary 2026-06-25)
_orig_sent_encode = _bsu.sent_encode
def _safe_sent_encode(tokenizer, sent):
    if getattr(tokenizer, 'model_max_length', 0) > 10_000:
        tokenizer.model_max_length = 512
    return _orig_sent_encode(tokenizer, sent)
_bsu.sent_encode = _safe_sent_encode

_bleu  = hf_evaluate.load('bleu')
_rouge = hf_evaluate.load('rouge')

def _compute_metrics(hyps, refs, variant, uncertain_policy='present'):
    cache = REPO_ROOT / 'reports' / f'eval_metrics_{variant}.json'
    if cache.exists():
        print(f'  [{variant}] loaded from cache')
        return json.loads(cache.read_text())

    print(f'  [{variant}] computing BERTScore…')
    _, _, F = _bert_score(hyps, refs,
        model_type=params['eval']['bertscore_model'],
        lang='en', device=DEVICE, verbose=False, batch_size=16)
    bertscore_f1 = float(F.mean())
    bertscore_per_study = F.tolist()

    print(f'  [{variant}] computing CheXbert…')
    hyp_labels = run_chexbert(hyps, uncertain_policy=uncertain_policy, device=DEVICE)
    ref_labels = run_chexbert(refs, uncertain_policy=uncertain_policy, device=DEVICE)
    micro_f1 = f1_score(ref_labels, hyp_labels, average='micro', zero_division=0)
    macro_f1 = f1_score(ref_labels, hyp_labels, average='macro', zero_division=0)
    per_label_f1 = {label: f1_score(ref_labels[:, i], hyp_labels[:, i], zero_division=0)
                   for i, label in enumerate(CHEXBERT_LABELS)}

    print(f'  [{variant}] computing BLEU/ROUGE…')
    bleu4  = _bleu.compute( predictions=hyps, references=[[r] for r in refs], max_order=4)['bleu']
    rouge_l = _rouge.compute(predictions=hyps, references=refs)['rougeL']

    result = {
        'variant': variant,
        'bertscore_f1': bertscore_f1,
        'chexbert_micro_f1': micro_f1,
        'chexbert_macro_f1': macro_f1,
        'bleu4': bleu4,
        'rouge_l': rouge_l,
        'per_label_f1': per_label_f1,
        'bertscore_per_study': bertscore_per_study,
    }
    cache.write_text(json.dumps(result, indent=2))
    print(f'  [{variant}] saved to {cache.name}')
    return result

metrics = {}
# Load zero-shot baseline from existing JSON if available
zs_baseline = REPO_ROOT / 'reports' / 'baseline_results.json'
if zs_baseline.exists():
    raw = json.loads(zs_baseline.read_text())
    m = raw['metrics']
    metrics['zero_shot'] = {
        'variant': 'zero_shot',
        'bertscore_f1':       m.get('bertscore_f1', float('nan')),
        'chexbert_micro_f1':  m.get('f1_chexbert_micro_present', float('nan')),
        'chexbert_macro_f1':  m.get('f1_chexbert_macro_present', float('nan')),
        'bleu4':              m.get('bleu4', float('nan')),
        'rouge_l':            m.get('rouge_l', float('nan')),
        'per_label_f1':       m.get('per_label_f1_present', {}),
        'bertscore_per_study': raw.get('bertscore_per_study', []),
    }
    print('[zero_shot] loaded from baseline_results.json')

for variant, hyps in hypotheses.items():
    if variant == 'zero_shot' and 'zero_shot' in metrics:
        continue  # already loaded above
    metrics[variant] = _compute_metrics(hyps, references, variant)

_bsu.sent_encode = _orig_sent_encode  # restore
print('\nDone. Variants evaluated:', list(metrics.keys()))

## STEP 4 — Summary table and figures

In [ ]:
DISPLAY_NAMES = {
    'zero_shot':   'Zero-shot MedGemma',
    'uniform_v3':  'QLoRA uniform (v3)',
    'weighted_v4': 'QLoRA weighted (v4)',
}

rows = []
for variant in ('zero_shot', 'uniform_v3', 'weighted_v4'):
    if variant not in metrics: continue
    m = metrics[variant]
    rows.append({
        'Model':             DISPLAY_NAMES.get(variant, variant),
        'BERTScore-F1':      round(m['bertscore_f1'],      4),
        'CheXbert micro-F1': round(m['chexbert_micro_f1'], 4),
        'CheXbert macro-F1': round(m['chexbert_macro_f1'], 4),
        'BLEU-4':            round(m['bleu4'],             4),
        'ROUGE-L':           round(m['rouge_l'],           4),
    })

summary_df = pd.DataFrame(rows).set_index('Model')
display(summary_df)

# ── Bar chart: BERTScore comparison ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = ['#7f7f7f', '#1f77b4', '#d62728']
names  = [DISPLAY_NAMES.get(v, v) for v in ('zero_shot', 'uniform_v3', 'weighted_v4') if v in metrics]

for ax, metric, title in [
    (axes[0], 'BERTScore-F1',      'BERTScore-F1 (primary metric)'),
    (axes[1], 'CheXbert micro-F1', 'CheXbert micro-F1 (diagnostic)'),
]:
    vals = [metrics[v][metric.lower().replace('-', '_').replace(' ', '_').replace('chexbert_micro_f1', 'chexbert_micro_f1').replace('bertscore_f1', 'bertscore_f1')] 
            for v in ('zero_shot', 'uniform_v3', 'weighted_v4') if v in metrics]
    bars = ax.bar(names, vals, color=colors[:len(vals)], alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.002, f'{val:.4f}',
                ha='center', va='bottom', fontsize=9)
    ax.set_ylim(0, max(vals) * 1.15)
    ax.set_title(title)
    ax.set_ylabel('Score')
    ax.tick_params(axis='x', labelrotation=15)

plt.suptitle('Test-set evaluation — zero-shot vs fine-tuned variants', fontsize=13)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'eval_metric_summary.png', dpi=150, bbox_inches='tight')
print('Saved eval_metric_summary.png')
plt.show()

# ── Per-label F1 heatmap ──────────────────────────────────────────────────────
label_rows = []
for variant in ('zero_shot', 'uniform_v3', 'weighted_v4'):
    if variant not in metrics: continue
    per_label = metrics[variant].get('per_label_f1', {})
    row = {'Model': DISPLAY_NAMES.get(variant, variant)}
    row.update({k: round(v, 3) for k, v in per_label.items()})
    label_rows.append(row)

if label_rows:
    label_df = pd.DataFrame(label_rows).set_index('Model')
    fig, ax = plt.subplots(figsize=(16, 2.5))
    im = ax.imshow(label_df.values, cmap='RdYlGn', vmin=0, vmax=0.6, aspect='auto')
    ax.set_xticks(range(len(label_df.columns)))
    ax.set_xticklabels(label_df.columns, rotation=35, ha='right', fontsize=8)
    ax.set_yticks(range(len(label_df.index)))
    ax.set_yticklabels(label_df.index, fontsize=9)
    for i in range(len(label_df.index)):
        for j in range(len(label_df.columns)):
            ax.text(j, i, f'{label_df.values[i,j]:.2f}', ha='center', va='center', fontsize=7)
    plt.colorbar(im, ax=ax, shrink=0.8, label='F1-CheXbert')
    ax.set_title('Per-label CheXbert F1 — zero-shot vs fine-tuned (diagnostic only)')
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'eval_per_label_f1.png', dpi=150, bbox_inches='tight')
    print('Saved eval_per_label_f1.png')
    plt.show()

## STEP 5 — Domain Shift Audit: Acquisition Shift

Simulates equipment-level domain shift (different manufacturer, kVp calibration, digitization quality)
via controlled image perturbations. Sweeps each perturbation type across its magnitude grid and plots
the BERTScore degradation curve.

**Perturbations:** brightness · contrast · gamma · gaussian_noise · jpeg_compression  
**Metric:** BERTScore-F1 (primary) — does not require CheXbert, cleaner signal for this audit.

In [ ]:
from src.domain_shift_audit.acquisition_shift import (
    run_acquisition_sweep, DEFAULT_MAGNITUDES
)

# Use the best fine-tuned variant for the audit
AUDIT_VARIANT = 'uniform_v3'   # switch to 'weighted_v4' once available

# Load model once for the whole audit
print(f'Loading {AUDIT_VARIANT} for acquisition audit…')
from src.training.model import load_model_and_processor
from peft import PeftModel

audit_model, audit_proc = load_model_and_processor(
    model_id=params['model']['base_model_id'],
    quantization=params['model']['quantization'],
)
adapter_path = CHECKPOINT_DIR / AUDIT_VARIANT / 'best_model'
if not (CHECKPOINT_DIR / AUDIT_VARIANT.replace('_', '_') / 'best_model').exists():
    # Try formatted path
    adapter_path = CHECKPOINT_DIR / f'qlora_{AUDIT_VARIANT}' / 'best_model'
audit_model = PeftModel.from_pretrained(audit_model, str(adapter_path))
audit_model.eval()

def _bertscore_metric_fn(perturbed_images):
    """Generate reports for perturbed images and return mean BERTScore-F1."""
    import bert_score.utils as _bsu2
    from bert_score import score as _bert_score2
    _bsu2.sent_encode = _safe_sent_encode  # apply patch

    hyps = []
    for img, (_, row) in zip(perturbed_images, test_df.iterrows()):
        indication = str(row.get('indication', '') or '').strip()
        if indication.lower() in {'nan', 'none', ''}: indication = ''
        user_text = f'Indication: {indication}\n{SYSTEM_PROMPT}' if indication else SYSTEM_PROMPT
        content = [{'type': 'image'}, {'type': 'text', 'text': user_text}]
        prompt = audit_proc.apply_chat_template(
            [{'role': 'user', 'content': content}],
            add_generation_prompt=True, tokenize=False,
        )
        inputs = audit_proc(text=prompt, images=[img], return_tensors='pt', padding=True)
        inputs = {k: v.to(audit_model.device) for k, v in inputs.items()}
        with torch.inference_mode():
            out = audit_model.generate(
                **inputs,
                max_new_tokens=params['model']['max_new_tokens'],
                do_sample=False,
                pad_token_id=audit_proc.tokenizer.eos_token_id,
            )
        hyp = audit_proc.tokenizer.decode(
            out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
        ).strip()
        hyps.append(hyp)

    _, _, F = _bert_score2(hyps, references,
        model_type=params['eval']['bertscore_model'],
        lang='en', device=DEVICE, verbose=False, batch_size=16)
    return float(F.mean())

# Load test images once
print('Loading test images…')
test_images = [_load_image(row) for _, row in test_df.iterrows()]
print(f'Loaded {len(test_images)} images')

# Run sweep for all 5 perturbation types
acq_results = {}
for perturb_type in DEFAULT_MAGNITUDES:
    cache = REPO_ROOT / 'reports' / f'acq_shift_{AUDIT_VARIANT}_{perturb_type}.json'
    if cache.exists():
        acq_results[perturb_type] = json.loads(cache.read_text())
        print(f'  {perturb_type}: loaded from cache')
    else:
        print(f'  {perturb_type}: running sweep…')
        res = run_acquisition_sweep(
            images=test_images,
            perturb_type=perturb_type,
            metric_fn=_bertscore_metric_fn,
            metric_name='bertscore_f1',
        )
        data = res.to_dataframe().to_dict(orient='list')
        cache.write_text(json.dumps(data))
        acq_results[perturb_type] = data
        print(f'  {perturb_type}: done — saved to {cache.name}')

del audit_model, audit_proc
torch.cuda.empty_cache()
print('VRAM freed.')

In [ ]:
# ── Acquisition shift degradation curves ─────────────────────────────────────
fig, axes = plt.subplots(1, len(acq_results), figsize=(4 * len(acq_results), 4), sharey=True)
if len(acq_results) == 1: axes = [axes]

for ax, (perturb_type, data) in zip(axes, acq_results.items()):
    df = pd.DataFrame(data)
    ax.plot(df['magnitude'], df['bertscore_f1'], marker='o', color='#1f77b4', linewidth=2)
    baseline = df[df['relative_degradation'].abs() < 0.001]['bertscore_f1'].iloc[0] if len(df) > 0 else None
    if baseline is not None:
        ax.axhline(baseline, color='gray', linestyle='--', linewidth=1, label=f'baseline={baseline:.4f}')
        ax.legend(fontsize=8)
    ax.set_title(perturb_type.replace('_', ' ').title())
    ax.set_xlabel('Magnitude')
    if ax == axes[0]: ax.set_ylabel('BERTScore-F1')
    ax.set_ylim(0.5, 0.80)

plt.suptitle(f'Acquisition Shift Audit — {AUDIT_VARIANT}', fontsize=13)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'eval_acquisition_shift.png', dpi=150, bbox_inches='tight')
print('Saved eval_acquisition_shift.png')
plt.show()

## STEP 6 — Domain Shift Audit: Prevalence Shift (Importance Sampling)

Estimates how model performance would change if the test set had a different label distribution —
e.g., higher Pleural Effusion or Atelectasis prevalence (plausible LATAM shift).

Method: importance sampling re-weighting  
  `w_i = π_target(y_i) / π_source(y_i)`  
  `weighted_metric = Σ w_i · score_i / Σ w_i`

ESS = (Σw)² / Σw² is reported at each point. ESS < 30 → estimate unreliable.

In [ ]:
from src.data.labels import CHEXBERT_LABELS

# Labels to sweep (clinically relevant LATAM shift candidates)
SHIFT_LABELS = ['Pleural Effusion', 'Atelectasis', 'Pneumonia', 'Consolidation']

test_label_matrix = test_df[CHEXBERT_LABELS].values.astype(float)

def _importance_weights(label_matrix, label_idx, pi_target, pi_source):
    y = label_matrix[:, label_idx]
    w = np.where(y == 1,
        pi_target / max(pi_source, 1e-9),
        (1 - pi_target) / max(1 - pi_source, 1e-9)
    )
    return w.astype(np.float64)

def _ess(weights):
    w = weights / weights.sum()
    return float(1.0 / (w**2).sum())

pi_targets = np.linspace(0.01, 0.50, 20)

prev_results = {}
for label in SHIFT_LABELS:
    label_idx  = CHEXBERT_LABELS.index(label)
    pi_source  = float(test_label_matrix[:, label_idx].mean())
    sweep_rows = []

    for variant in ('zero_shot', 'uniform_v3', 'weighted_v4'):
        if variant not in metrics: continue
        scores = np.array(metrics[variant].get('bertscore_per_study', []))
        if len(scores) != len(test_label_matrix): continue

        for pi_t in pi_targets:
            w = _importance_weights(test_label_matrix, label_idx, pi_t, pi_source)
            ess_val = _ess(w)
            w_metric = float(np.average(scores, weights=w))
            sweep_rows.append({
                'variant': variant,
                'target_prevalence': round(float(pi_t), 4),
                'weighted_bertscore_f1': w_metric,
                'ess': ess_val,
                'reliable': ess_val >= 30,
            })

    prev_results[label] = pd.DataFrame(sweep_rows)
    print(f'{label}: source prevalence = {pi_source:.3f}')

# ── Plot ─────────────────────────────────────────────────────────────────────
palette = {'zero_shot': '#7f7f7f', 'uniform_v3': '#1f77b4', 'weighted_v4': '#d62728'}
fig, axes = plt.subplots(1, len(SHIFT_LABELS), figsize=(5 * len(SHIFT_LABELS), 4), sharey=True)
if len(SHIFT_LABELS) == 1: axes = [axes]

for ax, label in zip(axes, SHIFT_LABELS):
    df = prev_results[label]
    for variant, color in palette.items():
        vdf = df[df['variant'] == variant]
        reliable = vdf[vdf['reliable']]
        unreliable = vdf[~vdf['reliable']]
        if len(reliable):
            ax.plot(reliable['target_prevalence'], reliable['weighted_bertscore_f1'],
                    color=color, linewidth=2, label=DISPLAY_NAMES.get(variant, variant))
        if len(unreliable):
            ax.plot(unreliable['target_prevalence'], unreliable['weighted_bertscore_f1'],
                    color=color, linewidth=1, linestyle=':', alpha=0.5)
    ax.set_xlabel(f'Target prevalence of\n{label}')
    ax.set_title(label)
    ax.legend(fontsize=7)
    ax.set_ylim(0.60, 0.80)

axes[0].set_ylabel('Weighted BERTScore-F1')
plt.suptitle('Prevalence Shift Audit — importance-sampling re-weighting\n(dashed = ESS < 30, estimate unreliable)', fontsize=12)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'eval_prevalence_shift.png', dpi=150, bbox_inches='tight')
print('Saved eval_prevalence_shift.png')
plt.show()

## STEP 7 — [SCAFFOLD] RAG Retrieval: Similar Report Injection

**Idea:** At inference time, retrieve the top-k most similar training reports and inject them
as few-shot examples in the prompt. The model then generates conditioned on:
1. The chest X-ray image
2. The clinical indication
3. k reference reports from visually/semantically similar cases

**Retrieval strategy (two options, both scaffolded below):**
- **Option A — TF-IDF on indication text:** fast, CPU-only, no GPU needed.
  Retrieves by textual similarity of the clinical indication ("shortness of breath + O2" → finds similar indications).
- **Option B — Sentence embeddings on findings:** uses `sentence-transformers`, captures report semantics.
  Retrieves by content similarity of existing reports (more expensive, better quality).

**This cell is CPU-only and can run now while training is in progress.**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ── Build retrieval index over training set ───────────────────────────────────
train_df = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
train_df = train_df[
    train_df['findings'].notna() & (train_df['findings'].str.strip() != '')
].reset_index(drop=True)

train_indications = train_df['indication'].fillna('').astype(str).tolist()
train_findings    = train_df['findings'].str.strip().tolist()

# Option A: TF-IDF on indications
tfidf = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
train_tfidf_matrix = tfidf.fit_transform(train_indications)  # (n_train, vocab)
print(f'TF-IDF index built: {train_tfidf_matrix.shape[0]} docs × {train_tfidf_matrix.shape[1]} features')

def retrieve_similar_reports(query_indication: str, k: int = 3) -> list[dict]:
    """Return the top-k training reports most similar to the query indication.

    Returns a list of dicts with keys: indication, findings, similarity.
    """
    query_vec = tfidf.transform([query_indication])
    sims = cosine_similarity(query_vec, train_tfidf_matrix).flatten()
    top_k_idx = sims.argsort()[::-1][:k]
    return [
        {
            'indication': train_indications[i],
            'findings':   train_findings[i],
            'similarity': float(sims[i]),
        }
        for i in top_k_idx
    ]

def build_rag_prompt(indication: str, retrieved: list[dict], k: int = 3) -> str:
    """Format the RAG-augmented prompt for MedGemma.

    The retrieved examples are injected as few-shot context before
    asking the model to generate the findings for the new study.
    """
    parts = []
    if retrieved:
        parts.append('The following are findings from similar studies for reference:')
        for i, r in enumerate(retrieved[:k], 1):
            parts.append(f'  Example {i} (similarity={r["similarity"]:.2f}):')
            if r['indication'].strip():
                parts.append(f'    Indication: {r["indication"].strip()}')
            parts.append(f'    Findings: {r["findings"].strip()}')
        parts.append('')  # blank line separator
    if indication.strip():
        parts.append(f'Indication: {indication.strip()}')
    parts.append('Findings:')
    return '\n'.join(parts)

# ── Smoke test ────────────────────────────────────────────────────────────────
query = 'Patient with shortness of breath and low oxygen saturation, history of smoking'
retrieved = retrieve_similar_reports(query, k=3)
print(f'\nQuery: "{query}"')
print(f'\nTop-3 retrieved:')
for r in retrieved:
    print(f'  sim={r["similarity"]:.3f} | ind: {r["indication"][:80]}')
    print(f'            | fnd: {r["findings"][:100]}…')
print(f'\nRAG prompt preview:')
print(build_rag_prompt(query, retrieved))

In [ ]:
# ── RAG inference stub (runs when a checkpoint is available) ──────────────────
# Uncomment and run after training completes to compare RAG vs standard prompting.

# RAG_VARIANT   = 'uniform_v3'
# RAG_K         = 3          # number of retrieved examples
# cache_rag     = REPO_ROOT / 'reports' / f'eval_hypotheses_rag_k{RAG_K}_{RAG_VARIANT}.json'

# rag_model, rag_proc = load_model_and_processor(
#     model_id=params['model']['base_model_id'],
#     quantization=params['model']['quantization'],
# )
# rag_model = PeftModel.from_pretrained(rag_model, str(CHECKPOINT_DIR / RAG_VARIANT / 'best_model'))
# rag_model.eval()

# rag_hyps = []
# for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
#     indication = str(row.get('indication', '') or '').strip()
#     if indication.lower() in {'nan', 'none', ''}: indication = ''
#     retrieved  = retrieve_similar_reports(indication, k=RAG_K)
#     user_text  = build_rag_prompt(indication, retrieved, k=RAG_K)
#     content    = [{'type': 'image'}, {'type': 'text', 'text': user_text}]
#     prompt     = rag_proc.apply_chat_template(
#         [{'role': 'user', 'content': content}],
#         add_generation_prompt=True, tokenize=False,
#     )
#     img    = _load_image(row)
#     inputs = rag_proc(text=prompt, images=[img], return_tensors='pt', padding=True)
#     inputs = {k: v.to(rag_model.device) for k, v in inputs.items()}
#     with torch.inference_mode():
#         out = rag_model.generate(**inputs,
#             max_new_tokens=params['model']['max_new_tokens'],
#             do_sample=False, pad_token_id=rag_proc.tokenizer.eos_token_id)
#     hyp = rag_proc.tokenizer.decode(
#         out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
#     rag_hyps.append(hyp)

# cache_rag.write_text(json.dumps(rag_hyps, ensure_ascii=False, indent=2))
# metrics[f'rag_k{RAG_K}_{RAG_VARIANT}'] = _compute_metrics(rag_hyps, references,
#     f'rag_k{RAG_K}_{RAG_VARIANT}')
# print(f"RAG BERTScore-F1: {metrics[f'rag_k{RAG_K}_{RAG_VARIANT}']['bertscore_f1']:.4f}")
# del rag_model, rag_proc; torch.cuda.empty_cache()

print('RAG scaffold ready. Uncomment the block above to run after training.')
print(f'TF-IDF index: {train_tfidf_matrix.shape[0]} training reports indexed.')

## STEP 8 — [SCAFFOLD] Association Rules Conditioner

**Idea:** Mine frequent co-occurrence patterns from the training label matrix
and use them to inject a soft diagnostic prior into the prompt at inference time.

**Mechanism:**
1. Compute pairwise conditional probabilities P(label_B | label_A) from training labels.
2. At inference time, use a lightweight zero-shot or keyword pass on the indication to estimate
   which labels might be present.
3. Look up high-confidence rules for those labels and format them as a soft conditioning hint:
   > *"Clinical note: in studies with similar presentations, Pleural Effusion co-occurs with
     Edema in 61% of cases and with Cardiomegaly in 33% of cases."*
4. Inject this hint into the prompt before the Findings tag.

**This cell is CPU-only and can run now while training is in progress.**

In [ ]:
from src.data.labels import CHEXBERT_LABELS

# ── Build association rule table from training labels ─────────────────────────
train_label_matrix = train_df[CHEXBERT_LABELS].values.astype(float)
n = len(train_label_matrix)

# Pairwise conditional: P(B | A) = P(A ∩ B) / P(A)
co_matrix    = (train_label_matrix.T @ train_label_matrix) / n   # (14,14) joint prob
prevalences  = train_label_matrix.mean(axis=0)                    # (14,) marginal

rules = []
for i, label_a in enumerate(CHEXBERT_LABELS):
    if prevalences[i] < 0.01: continue  # skip near-zero prevalence antecedents
    for j, label_b in enumerate(CHEXBERT_LABELS):
        if i == j: continue
        if prevalences[j] < 0.01: continue
        p_ab = co_matrix[i, j]
        p_a  = prevalences[i]
        p_b  = prevalences[j]
        if p_a == 0: continue
        conf  = p_ab / p_a           # P(B | A)
        lift  = conf / p_b if p_b > 0 else 0
        rules.append({
            'antecedent':  label_a,
            'consequent':  label_b,
            'support':     round(float(p_ab),  4),
            'confidence':  round(float(conf),  4),
            'lift':        round(float(lift),  4),
            'p_antecedent': round(float(p_a),  4),
        })

rules_df = pd.DataFrame(rules).sort_values('confidence', ascending=False)
print(f'Total rules: {len(rules_df)}')
print(f'High-confidence rules (conf ≥ 0.30):')
display(rules_df[rules_df['confidence'] >= 0.30].head(20))

In [ ]:
# ── Prompt conditioner ────────────────────────────────────────────────────────

# Keyword → likely antecedent label mapping (extend as needed)
_INDICATION_KEYWORDS = {
    'Cardiomegaly':     ['cardio', 'cardiac', 'heart', 'cardiomeg'],
    'Pleural Effusion': ['effusion', 'pleural', 'fluid'],
    'Pneumonia':        ['pneumonia', 'infection', 'consolidation', 'fever'],
    'Edema':            ['edema', 'pulmonary edema', 'fluid overload', 'chf'],
    'Atelectasis':      ['atelectasis', 'collapse', 'post-op'],
    'Pneumothorax':     ['pneumothorax', 'ptx', 'chest pain', 'trauma'],
    'Support Devices':  ['intubated', 'icu', 'ventilated', 'post-op', 'line', 'tube'],
}

def infer_likely_labels(indication: str, threshold: float = 0.0) -> list[str]:
    """Return labels whose keywords appear in the indication text."""
    indication_lower = indication.lower()
    return [
        label for label, keywords in _INDICATION_KEYWORDS.items()
        if any(kw in indication_lower for kw in keywords)
    ]

def build_association_hint(
    likely_labels: list[str],
    rules_df: pd.DataFrame,
    min_confidence: float = 0.25,
    max_rules: int = 4,
) -> str:
    """Format a soft diagnostic prior from association rules.

    Returns empty string if no high-confidence rules apply.
    """
    if not likely_labels:
        return ''
    relevant = rules_df[
        rules_df['antecedent'].isin(likely_labels) &
        (rules_df['confidence'] >= min_confidence)
    ].sort_values('confidence', ascending=False).head(max_rules)

    if relevant.empty:
        return ''

    lines = ['Clinical association context (from training set statistics):']
    for _, rule in relevant.iterrows():
        lines.append(
            f'  - {rule["antecedent"]} co-occurs with {rule["consequent"]} '
            f'in {100*rule["confidence"]:.0f}% of similar studies '
            f'(lift={rule["lift"]:.2f}).'
        )
    return '\n'.join(lines)

def build_conditioned_prompt(
    indication: str,
    rules_df: pd.DataFrame,
    min_confidence: float = 0.25,
) -> str:
    """Full prompt with association rule soft conditioner injected."""
    likely_labels = infer_likely_labels(indication)
    hint = build_association_hint(likely_labels, rules_df, min_confidence=min_confidence)
    parts = []
    if hint:
        parts.append(hint)
        parts.append('')
    if indication.strip():
        parts.append(f'Indication: {indication.strip()}')
    parts.append('Findings:')
    return '\n'.join(parts)

# ── Smoke test ────────────────────────────────────────────────────────────────
test_indications = [
    'Shortness of breath, rule out pleural effusion',
    'Chest pain, history of CHF and pulmonary edema',
    'Post-op day 1, patient intubated in ICU',
    'Fever, cough, possible pneumonia',
    'Routine chest X-ray, no specific indication',
]
for ind in test_indications:
    print(f'=== Indication: "{ind}" ===')
    print(build_conditioned_prompt(ind, rules_df))
    print()

In [ ]:
# ── Conditioned inference stub (runs after training completes) ─────────────────
# Combines RAG + association rules into one augmented prompt.

# COND_VARIANT = 'uniform_v3'
# COND_K_RAG   = 3
# COND_MIN_CONF = 0.25
# cache_cond = REPO_ROOT / 'reports' / f'eval_hypotheses_conditioned_{COND_VARIANT}.json'

# cond_model, cond_proc = load_model_and_processor(
#     model_id=params['model']['base_model_id'],
#     quantization=params['model']['quantization'],
# )
# cond_model = PeftModel.from_pretrained(cond_model, str(CHECKPOINT_DIR / COND_VARIANT / 'best_model'))
# cond_model.eval()

# cond_hyps = []
# for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
#     indication = str(row.get('indication', '') or '').strip()
#     if indication.lower() in {'nan', 'none', ''}: indication = ''

#     # Build augmented prompt: rules conditioner + RAG examples
#     retrieved = retrieve_similar_reports(indication, k=COND_K_RAG)
#     cond_hint = build_conditioned_prompt(indication, rules_df, COND_MIN_CONF)
#     rag_part  = build_rag_prompt('', retrieved, k=COND_K_RAG)   # no indication in RAG part
#     user_text = f'{cond_hint}\n{rag_part}' if cond_hint else rag_part

#     content = [{'type': 'image'}, {'type': 'text', 'text': user_text}]
#     prompt  = cond_proc.apply_chat_template(
#         [{'role': 'user', 'content': content}],
#         add_generation_prompt=True, tokenize=False,
#     )
#     img    = _load_image(row)
#     inputs = cond_proc(text=prompt, images=[img], return_tensors='pt', padding=True)
#     inputs = {k: v.to(cond_model.device) for k, v in inputs.items()}
#     with torch.inference_mode():
#         out = cond_model.generate(**inputs,
#             max_new_tokens=params['model']['max_new_tokens'],
#             do_sample=False, pad_token_id=cond_proc.tokenizer.eos_token_id)
#     hyp = cond_proc.tokenizer.decode(
#         out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
#     cond_hyps.append(hyp)

# cache_cond.write_text(json.dumps(cond_hyps, ensure_ascii=False, indent=2))
# metrics[f'conditioned_{COND_VARIANT}'] = _compute_metrics(cond_hyps, references,
#     f'conditioned_{COND_VARIANT}')
# print(f"Conditioned BERTScore-F1: {metrics[f'conditioned_{COND_VARIANT}']['bertscore_f1']:.4f}")
# del cond_model, cond_proc; torch.cuda.empty_cache()

print('Conditioned inference scaffold ready.')
print(f'Association rules mined: {len(rules_df)} rules')
print(f'High-confidence rules (≥30%): {(rules_df.confidence >= 0.30).sum()}')

## Summary

| Step | Status | Output |
|------|--------|--------|
| 1. Environment | ✓ always | — |
| 2. Generate hypotheses | requires GPU + checkpoint | `reports/eval_hypotheses_{variant}.json` |
| 3. Metric stack | requires 2 | `reports/eval_metrics_{variant}.json` |
| 4. Summary table | requires 3 | `reports/figures/eval_metric_summary.png` |
| 5. Acquisition shift | requires GPU + checkpoint | `reports/figures/eval_acquisition_shift.png` |
| 6. Prevalence shift | requires step 3 | `reports/figures/eval_prevalence_shift.png` |
| 7. RAG scaffold | ✓ CPU-only | `retrieve_similar_reports()`, `build_rag_prompt()` |
| 8. Assoc. rules scaffold | ✓ CPU-only | `rules_df`, `build_conditioned_prompt()` |

**STEPs 7–8 are fully runnable now** (CPU, no checkpoint required).  
To run the conditioned inference (STEP 8 stub), uncomment the inference block after training completes.